# Dataset Explore

In [1]:
from datasets import load_dataset
from tqdm.notebook import tqdm

In [4]:
# dataset = load_dataset("bnadimi/PyraNet-Verilog", split = "train")
dataset = load_dataset('json', data_files='.local_models/lora_model_2025-10-27T17:31:20.129653/dataset.json', split = "train")

In [6]:
len(dataset['text'])

53000

In [10]:
min_length = [len(dataset['text']), 0]
max_length = [len(dataset['text']), 0]
for i in range(1, len(dataset['text'])):
    cur_length = len(dataset['text'][i])
    if cur_length < min_length[0]:
        min_length[0] = cur_length
        min_length[1] = i
    if cur_length > max_length[0]:
        max_length[0] = cur_length
        max_length[1] = i
print(min_length, max_length)

[990, 9722] [98392, 5995]


## Dataset 350k

In [2]:
dataset_350 = load_dataset('json', data_files='pyranet-no-error.json', split = "train")

In [3]:
len(dataset_350)

357715

In [4]:
dataset_350['code'][15984]

"module pc_Mux(PCsrc, potNext1, potNext2, PC_next);\n\ninput PCsrc;\ninput [31:0] potNext1;\ninput [31:0] potNext2;\n\noutput [31:0] PC_next;\n/*\nalways @(PCsrc) begin\ncase(PCsrc)\n\n1'b0: PC_next <= potNext1; //+4\n1'b1: PC_next <= potNext2; //Jump\n\nendcase\nend\n*//*\nalways @(PCsrc) begin\n\nif(PCsrc == 1) begin\nPC_next <= potNext2;\nend else begin\nPC_next <= potNext1;\nend\n\nend*/\n\nassign PC_next= PCsrc?potNext2:potNext1;\n\nendmodule "

In [14]:
del_elements = {20053, 72641, 74745, 78061, 122269, 158568, 198975, 209047, 258517, 293947}
all_dataset_element = list(range(len(dataset_350)))
for element in del_elements:
    all_dataset_element.remove(element)
dataset_350 = dataset_350.select(all_dataset_element)
len(dataset_350)

357705

In [3]:
import json, re, tempfile, subprocess, os
import numpy as np

def module_extraction(example_code, format_code=False):
    if format_code:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".v") as fp:
            fp.write(example_code.encode(encoding='utf-8'))
            fp.close()
            sub_run = subprocess.run(['verible-verilog-format', fp.name], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            
            # example_code = sub_run.stdout.decode("utf-8")
            with open(fp.name, 'r') as file:
                example_code = file.read()
    
        subprocess.run(['rm', '-f', fp.name])
    code_lines = example_code.splitlines()
    regex_stage = 0
    regexes = [r'^.*module\s+', r'^.*module$']
    
    module_def_span = np.array([None, None, None, None])
    num_indexs = 0
    comment_stage = 0
    multiline_comment = [False, -1]
    for i in range(len(code_lines)):
        line = code_lines[i]
        num_indexs += (i > 0) # plus previous \n
        if comment_stage == 0:
            line_for_comment = str(line)
            comment_ranges = []
            comment_num_indexs = 0
            while True:
                double_slash_index = line_for_comment.find("//")
                double_slash_index = [double_slash_index, double_slash_index]
                
                single_slash_index = line_for_comment.find("/*")
                single_slash_index = [single_slash_index, single_slash_index]
                
                single_slash_end_index = line_for_comment.rfind("*/")
                single_slash_end_index = [single_slash_end_index, single_slash_end_index]

                #
                # Handle for case of multi-line comment
                if multiline_comment[0]:
                    if (double_slash_index[0] != -1 and single_slash_end_index[0] != -1):
                        if (double_slash_index[0] < single_slash_end_index[0]) or (double_slash_index[0] - single_slash_end_index[0] == 1):
                            double_slash_index[0] = -1
                        # else: No else!!!                            
                else:
                    if double_slash_index[0] != -1 and single_slash_index[0] != -1: # fix all cases of comment!!
                        if double_slash_index[0] < single_slash_index[0]:
                            single_slash_index[0] = -1
                        else:
                            double_slash_index[0] = -1
                    if (double_slash_index[0] != -1 and single_slash_end_index[0] != -1):
                        if double_slash_index[0] < single_slash_end_index[0]:
                            single_slash_end_index[0] = -1
                    if single_slash_end_index[0] > -1 and single_slash_index[0] > -1 and (single_slash_end_index[0] - single_slash_index[0] == 1):
                        single_slash_end_index[0] = single_slash_end_index[1] = -1
                #
                # handle for start comment
                # start_comment_index = double_slash_index if double_slash_index != None else single_slash_index if single_slash_index != None else None
                if multiline_comment[0]:
                    if single_slash_end_index[0] != -1:
                        multiline_comment[0] = False

                        len_of_old_line = len(line_for_comment)
                        comment_ranges.append([0, single_slash_end_index[0] + 1])
                        comment_ranges[-1][0] += comment_num_indexs
                        comment_ranges[-1][1] += comment_num_indexs
                        
                        line_for_comment = line_for_comment[single_slash_end_index[0]+2:]
                        comment_num_indexs += len_of_old_line - len(line_for_comment)
                    elif multiline_comment[1] != i:
                        len_of_old_line = len(line_for_comment)
                        if len_of_old_line:
                            comment_ranges.append([0, len_of_old_line - 1])
                            comment_ranges[-1][0] += comment_num_indexs
                            comment_ranges[-1][1] += comment_num_indexs
                            
                            line_for_comment = line_for_comment[:0]
                            comment_num_indexs += len_of_old_line - len(line_for_comment)
                        break
                
                if not multiline_comment[0]:
                    if double_slash_index[0] != -1:
                        len_of_old_line = len(line_for_comment)
                        comment_ranges.append([double_slash_index[0], len_of_old_line - 1])
                        comment_ranges[-1][0] += comment_num_indexs
                        comment_ranges[-1][1] += comment_num_indexs
                        
                        line_for_comment = line_for_comment[:double_slash_index[0]]
                        comment_num_indexs += len_of_old_line - len(line_for_comment)
                    if single_slash_index[0] != -1 and single_slash_end_index[0] == -1:
                        multiline_comment[0] = True
                        multiline_comment[1] = i

                        len_of_old_line = len(line_for_comment)
                        comment_ranges.append([single_slash_index[0], len_of_old_line - 1])
                        comment_ranges[-1][0] += comment_num_indexs
                        comment_ranges[-1][1] += comment_num_indexs
                        
                        line_for_comment = line_for_comment[:single_slash_index[0]]
                        comment_num_indexs += len_of_old_line - len(line_for_comment)
                        
                    elif single_slash_index[0] != -1 and single_slash_end_index[0] != -1:
                        len_of_old_line = len(line_for_comment)
                        comment_ranges.append([single_slash_index[0], single_slash_end_index[0]+1])
                        comment_ranges[-1][0] += comment_num_indexs
                        comment_ranges[-1][1] += comment_num_indexs
                        
                        line_for_comment = line_for_comment[:single_slash_index[0]] + line_for_comment[single_slash_end_index[0]+2:]
                        comment_num_indexs += len_of_old_line - len(line_for_comment)
                        
                if double_slash_index[1] != -1 or single_slash_index[1] != -1 or single_slash_end_index[1] != -1:
                    continue
                else:
                    break
                
        # if len(comment_ranges):
        #     print(comment_ranges, i)

        section_start_idx = 0
        cur_section = ''

        if len(comment_ranges): # sort for correct order
            comment_ranges.sort(key=lambda item: item[0])
            
        while section_start_idx < len(line):
            old_section_start_idx = section_start_idx
            if len(comment_ranges):
                cur_section = line[section_start_idx:comment_ranges[0][0]]
                section_start_idx += comment_ranges[0][1] + 1
                comment_ranges.pop(0)
            else:
                cur_section = line[section_start_idx:]
                section_start_idx += len(cur_section)
        
            #
            if regex_stage == 0:
                for regex in regexes:
                    module_match = re.match(regex, cur_section)
                    if module_match != None:
                        regex_stage += 1
                        span = module_match.span()
                        module_def_span = np.vstack((module_def_span, [span[0] + num_indexs, span[1] + num_indexs, None, None]))
                        break
            if regex_stage == 1:
                semicon_pos = cur_section.find(";")
                if semicon_pos != -1:
                    module_def_span[-1][-2] = num_indexs + semicon_pos + 1
                    regex_stage += 1
            if regex_stage == 2:
                find_start = 0
                while True:
                    endmodule_word = "endmodule"
                    endmodule_pos = cur_section.find(endmodule_word, find_start)
                    endmodule_pos_end = endmodule_pos + len(endmodule_word)
                    if endmodule_pos != -1:
                        left_endmodule = False
                        right_endmodule = False

                        #
                        if endmodule_pos == 0:
                            left_endmodule = True
                        elif endmodule_pos > 0:
                            left_ord = ord(cur_section[endmodule_pos - 1] )
                            if not ((left_ord >= 65 and left_ord <= 90) or (left_ord >= 97 and left_ord <= 122) or left_ord == 95):
                                left_endmodule = True

                        #
                        if endmodule_pos_end < len(cur_section):
                            right_ord = ord(cur_section[endmodule_pos_end])
                            if not ((right_ord >= 65 and right_ord <= 90) or (right_ord >= 97 and right_ord <= 122) or right_ord == 95):
                                right_endmodule = True
                        elif endmodule_pos_end == len(cur_section):
                            right_endmodule = True

                        if left_endmodule and right_endmodule:
                            module_def_span[-1][-1] = num_indexs + endmodule_pos + 1
                            regex_stage = 0
                            break
                        else:
                            find_start += endmodule_pos_end
                    else:
                        break
                            # if left_ord == 13 or left_ord == 10 or left_ord == 9 or left_ord == 12 or left_ord == 32:
                            #     left_endmodule = True
                        # module_def_span[-1][-1] = num_indexs + endmodule_pos + 1
                        # regex_stage = 0
    
            num_indexs += section_start_idx - old_section_start_idx
            
    if len(module_def_span.shape) < 2:
        error_txt = f"Module span is all null {len(example_code)}: {example_code}"
        
        raise Exception(error_txt)
        
    module_def_span = np.delete(module_def_span, 0, 0)
    module_wrapper = ""
    module_definition = []
    module_output_code = []
    for span in module_def_span:
        if None in span:
            raise Exception(f"Span Error!! {len(example_code)}: {example_code}")
        module_definition += [example_code[span[0]:span[2]]]
        module_output_code += [example_code[span[2]:span[3]+8]]
        
    module_definition_with_upper_comments = []
    for i in range(len(module_def_span)):
        span = module_def_span[i]
        if i == 0:
            span[0] = 0
        else:
            span[0] = module_def_span[i-1][-1] + 1
        module_definition_with_upper_comments += [example_code[span[0]:span[2]]]

    return (module_definition, module_output_code, module_definition_with_upper_comments)
# code = """"""
# [module_definition, module_output_code, module_definition_with_upper_comments] = module_extraction(code)
# print(code)

In [4]:
from tqdm.notebook import tqdm
pbar = tqdm(total=len(dataset_350))

  0%|          | 0/357715 [00:00<?, ?it/s]

In [ ]:

for i in range(len(dataset_350)):
    example_codes = [dataset_350["code"][i]]
    example_descriptions = [dataset_350["description"][i]]
    for example_description, example_code in zip(example_descriptions, example_codes):
        # format, for \n placements
        example_code = example_code.replace("\\\\", "\\")
        # print(example_description)
        example_description = example_description.replace("\\\\", "\\")
        example_description_dict = json.loads(example_description)
        
        [module_definitions, module_output_codes, _] = module_extraction(example_code)
        
        #
        fullcode = ''
        for module_definition, module_output_code in zip(module_definitions, module_output_codes):
            fullcode += module_definition + module_output_code
    pbar.update(1)


In [6]:
pbar.close()

In [7]:
dataset_350['code'][33462]

'`timescale 1ns / 1ps\n//////////////////////////////////////////////////////////////////////////////////\n// Company: \n// Engineer: \n// \n// Create Date:    05:18:45 02/19/2022 \n// Design Name: \n// Module Name:    xor64 \n// Project Name: \n// Target Devices: \n// Tool versions: \n// Description: \n//\n// Dependencies: \n//\n// Revision: \n// Revision 0.01 - File Created\n// Additional Comments: \n//\n//////////////////////////////////////////////////////////////////////////////////\nmodule xor64( A, B, S);\n\tinput [63:0] A;\n\tinput [63:0] B;\n\toutput [63:0] S;\n\t \n\tassign S = A ^ B;\n\n\nendmodule\n'

In [4]:
def formatting_prompts_func(examples):
    example_descriptions = examples["description"]
    example_codes        = examples["code"]
    # texts = []
    instruction_collect = []
    module_def_collect = []
    output_collect = []
    fullcode_collec = []
    module_def_with_comments_collect = []
    for example_description, example_code in zip(example_descriptions, example_codes):
        # format, for \n placements
        example_code = example_code.replace("\\\\", "\\")
        
        example_description = example_description.replace("\\\\", "\\")
        example_description_dict = json.loads(example_description)
        
        [module_definitions, module_output_codes, module_definition_with_upper_comments] = module_extraction(example_code)
        instruction_collect.append(example_description)
        module_def_collect.append(module_definitions)
        output_collect.append(module_output_codes)
        module_def_with_comments_collect.append(module_definition_with_upper_comments)
        #
        # fullcode = ''
        # for module_definition, module_output_code, module_definition_with_upper_comment in zip(module_definitions, module_output_codes, module_definition_with_upper_comments):
            # fullcode += module_definition + module_output_code
            
        # fullcode_collec.append(fullcode)
    return { "instruction": instruction_collect,
             "module_definition": module_def_collect,
             "module_definition_with_comment": module_def_with_comments_collect,
             'module_code': output_collect
           }
pass

dataset_350 = dataset_350.map(formatting_prompts_func, batched = True, num_proc=os.cpu_count())

Map (num_proc=24):   0%|          | 0/357715 [00:00<?, ? examples/s]

In [ ]:
dataset_350.keys()

## Code compile

In [8]:
# dataset_iter = dataset_350.iter(batch_size=1)
from tqdm.notebook import tqdm
ErrorCompileItem = []
pbar = tqdm(total=len(dataset_350))
for i in range(len(dataset_350)):
# for example in dataset_iter:
    # example
    example_module_definitions = dataset_350["module_definition"][i]
    example_outputs        = dataset_350["module_code"][i]
    # print(example_module_definitions)
    # print(example_outputs)
    fullcode = ""
    for example_module_definition, example_output in zip(example_module_definitions, example_outputs):
        fullcode += example_module_definition + example_output + "\n"
        # print(fullcode)
    with tempfile.NamedTemporaryFile(delete=False, suffix=".v") as fp:
        fp.write(fullcode.encode(encoding='utf-8'))
        fp.close()
        outFilename = fp.name + '.out'
        sub_run = subprocess.run(['iverilog', fp.name, '-o', outFilename], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        if sub_run.returncode != 0:
            ErrorCompileItem.append([i, fp.name])
            print(i, fp.name)
    if sub_run.returncode == 0:
        subprocess.run(['rm', '-f', fp.name, outFilename])
    #     # example_code = sub_run.stdout.decode("utf-8")
    #     with open(fp.name, 'r') as file:
    #         example_code = file.read()
    pbar.update(1)
pbar.close()
ErrorCompileItem

  0%|          | 0/357715 [00:00<?, ?it/s]

994 /tmp/tmp0l3z6lj9.v
1003 /tmp/tmpm37bff3y.v
3363 /tmp/tmpyuugwl6v.v
3418 /tmp/tmp9p0kscyu.v
3429 /tmp/tmp_snosgvv.v
3834 /tmp/tmp3n5bv9sj.v
3991 /tmp/tmpx_26mky2.v
4000 /tmp/tmp2kv83g2q.v
4145 /tmp/tmp5wydp69i.v
4212 /tmp/tmp4_8_j890.v
4295 /tmp/tmpuhf39yof.v
4443 /tmp/tmpyv1q_y_1.v
4572 /tmp/tmpqijryb57.v
4765 /tmp/tmp2fsnt6te.v
4968 /tmp/tmpsjgir610.v
5044 /tmp/tmpjdj7f206.v
5100 /tmp/tmp5trvb32e.v
5224 /tmp/tmp9bd0vzsf.v
5345 /tmp/tmpsnplepax.v
5398 /tmp/tmpmc5cdus4.v
5475 /tmp/tmptcmv_ps1.v
5545 /tmp/tmpfvm4xeb9.v
5621 /tmp/tmp1kkcsm8v.v
5924 /tmp/tmp_5l4loq9.v
6253 /tmp/tmpc1acqt0k.v
6464 /tmp/tmpvvmm9hvr.v
6489 /tmp/tmpt_9wahvx.v
6519 /tmp/tmpf51ibo30.v
6999 /tmp/tmp4__bekzn.v
7106 /tmp/tmp38p5ylpd.v
7234 /tmp/tmpbkg7c45g.v
7241 /tmp/tmpyeduf4_a.v
7329 /tmp/tmps0h60skv.v
7345 /tmp/tmpipfbqfw5.v
7392 /tmp/tmpjzrflrr7.v
7498 /tmp/tmp8z4x7p6f.v
7506 /tmp/tmpeycx1yk3.v
7692 /tmp/tmp5iek7ive.v
7851 /tmp/tmpsnty5bik.v
8009 /tmp/tmp7nl1ym_4.v
8079 /tmp/tmpfl4ajgcp.v
8141 /tmp/tmpghuo

[[994, '/tmp/tmp0l3z6lj9.v'],
 [1003, '/tmp/tmpm37bff3y.v'],
 [3363, '/tmp/tmpyuugwl6v.v'],
 [3418, '/tmp/tmp9p0kscyu.v'],
 [3429, '/tmp/tmp_snosgvv.v'],
 [3834, '/tmp/tmp3n5bv9sj.v'],
 [3991, '/tmp/tmpx_26mky2.v'],
 [4000, '/tmp/tmp2kv83g2q.v'],
 [4145, '/tmp/tmp5wydp69i.v'],
 [4212, '/tmp/tmp4_8_j890.v'],
 [4295, '/tmp/tmpuhf39yof.v'],
 [4443, '/tmp/tmpyv1q_y_1.v'],
 [4572, '/tmp/tmpqijryb57.v'],
 [4765, '/tmp/tmp2fsnt6te.v'],
 [4968, '/tmp/tmpsjgir610.v'],
 [5044, '/tmp/tmpjdj7f206.v'],
 [5100, '/tmp/tmp5trvb32e.v'],
 [5224, '/tmp/tmp9bd0vzsf.v'],
 [5345, '/tmp/tmpsnplepax.v'],
 [5398, '/tmp/tmpmc5cdus4.v'],
 [5475, '/tmp/tmptcmv_ps1.v'],
 [5545, '/tmp/tmpfvm4xeb9.v'],
 [5621, '/tmp/tmp1kkcsm8v.v'],
 [5924, '/tmp/tmp_5l4loq9.v'],
 [6253, '/tmp/tmpc1acqt0k.v'],
 [6464, '/tmp/tmpvvmm9hvr.v'],
 [6489, '/tmp/tmpt_9wahvx.v'],
 [6519, '/tmp/tmpf51ibo30.v'],
 [6999, '/tmp/tmp4__bekzn.v'],
 [7106, '/tmp/tmp38p5ylpd.v'],
 [7234, '/tmp/tmpbkg7c45g.v'],
 [7241, '/tmp/tmpyeduf4_a.v'],
 [7329, '

In [9]:
len(ErrorCompileItem)

4618

In [45]:
dataset_350_ori["code"][564]

"module FullAdder(\n\tinput a, b, cin,\n\toutput sum, cout\n);\n\tassign sum = a^b^cin;\n\tassign cout = (a&b)|(b&cin)|(cin&a);\nendmodule\n\nmodule RippleCarryFA( // 4-bit version\n\tinput  [3:0] a, b,\n\toutput [3:0] sum,\n\toutput cout\n);\n\twire w0, w1, w2;\n\tFullAdder u0( .a(a[0]), .b(b[0]), .cin(1'b0), .sum(sum[0]), .cout(w0)   );\n\tFullAdder u1( .a(a[1]), .b(b[1]), .cin(w0),   .sum(sum[1]), .cout(w1)   );\n\tFullAdder u2( .a(a[2]), .b(b[2]), .cin(w1),   .sum(sum[2]), .cout(w2)   );\n\tFullAdder u3( .a(a[3]), .b(b[3]), .cin(w2),   .sum(sum[3]), .cout(cout) );\nendmodule\n"

In [38]:
len(ErrorCompileItem)

1

In [35]:
import json
with open("ErrorCompileItem.json", 'w+') as file:
    json.dump(list(ErrorCompileItem), file)

In [ ]:
for i in ErrorCompileItem:
    print(i)
    print(dataset_350["module_definition"][i][0] + dataset_350["output"][i][0])
    input()

# 

In [ ]:
!CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install llama-cpp-python

In [ ]:
!conda install -c conda-forge gcc=12.1.0 -y

In [1]:
from langchain_community.llms import LlamaCpp
from langchain_core.callbacks import CallbackManager, StreamingStdOutCallbackHandler
from langchain_core.prompts import PromptTemplate

In [2]:
template = """Below is an instruction that describes a Verilog module, paired with a module definition that provides further context for input and output ports. Write a response that only completes the code of Verilog module, inside XML tags and without other descriptions.

### Instruction:
{instruction}

Place the completion of the Verilog module in an XML tag <hdls></hdls>.
Inside the XML tag, all partial modules constructing the Verilog module, are placed inside a child XML tag <hdl></hdl>.
In each partial tag, the tag <module_definition></module_definition> is used to place the module definition of that module and
the tag <module_code></module_code> is used to place the module code of that module.

### Module Definition:
{module_def}

### Response:
"""

prompt = PromptTemplate.from_template(template)

In [3]:
# Callbacks support token-wise streaming
callback_manager = CallbackManager([StreamingStdOutCallbackHandler()])

In [4]:
n_gpu_layers = -1  # The number of layers to put on the GPU. The rest will be on the CPU. If you don't know how many layers there are, you can use -1 to move all to GPU.
n_batch = 512  # Should be between 1 and n_ctx, consider the amount of VRAM in your GPU.

# Make sure the model path is correct for your system!
llm = LlamaCpp(
    model_path="/home/thanh/vllm/GNU_COMBA/src/codellama-7b.Q4_K_M.gguf",
    n_gpu_layers=n_gpu_layers,
    n_batch=n_batch,
    # callback_manager=callback_manager,
    # verbose=True,  # Verbose is required to pass to the callback manager
)

ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    no
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA GeForce RTX 4060 Ti, compute capability 8.9, VMM: yes
llama_model_load_from_file_impl: using device CUDA0 (NVIDIA GeForce RTX 4060 Ti) - 15727 MiB free
llama_model_loader: loaded meta data with 34 key-value pairs and 291 tensors from /home/thanh/vllm/GNU_COMBA/src/codellama-7b.Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = lora_GGUF
llama_model_loader: - kv   3:                       general.quantized_by str              = Unsloth
llama_model_loader: - kv   4:               

In [5]:
llm_chain = prompt | llm
instruction = """Implement the Verilog module based on the following description. Assume that signals are positive clock/clk triggered unless otherwise stated.

Build a circuit with no inputs and one output. That output should always
drive 1 (or logic high)."""
module_def = """module TopModule (
  output one
);"""
llm_chain.invoke({"instruction": instruction, "module_def": module_def},)

llama_perf_context_print:        load time =     206.96 ms
llama_perf_context_print: prompt eval time =     206.78 ms /   237 tokens (    0.87 ms per token,  1146.16 tokens per second)
llama_perf_context_print:        eval time =    1193.54 ms /    70 runs   (   17.05 ms per token,    58.65 tokens per second)
llama_perf_context_print:       total time =    1426.73 ms /   307 tokens
llama_perf_context_print:    graphs reused =         67


"<hdls>\n<hdl>\n<module_definition>\nmodule TopModule (\n  output one\n);\n</module_definition>\n<module_code>\n\n   assign one = 1'b1;\nendmodule\n\n</module_code>\n</hdl>\n</hdls>"

In [7]:
llm.client.close()